# 12 — Modèle neuronal (MLP) pour la décorrélation + blend

On a prouvé que nos GBDT sont trop corrélés (0.89-0.93) pour qu'un blend aide. Un réseau de
neurones a une nature TRÈS différente d'un arbre -> il peut être décorrélé même s'il est un
peu plus faible seul. Test : corr(CatBoost, MLP) et est-ce que blend > 0.3607 (last fold) ?

MLP = scikit-learn (StandardScaler + clip + MLPClassifier). Boussole : LB ≈ last − 0.004.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.blending import rank_average
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6
WINDOWS = (5, 10, 20)
SMOOTHING = 30

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def feats_train(df, ref):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    return X

def feats_apply(df, ref):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

def clean(X):
    return X.replace([np.inf, -np.inf], np.nan)

def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

def make_mlp():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        FunctionTransformer(lambda x: np.clip(x, -5, 5)),
        MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, batch_size=4096,
                      learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                      n_iter_no_change=6, random_state=42),
    )

## CV : CatBoost (A) vs MLP (D) + corrélation + blend

In [ ]:
oof_A = np.zeros(len(train)); oof_D = np.zeros(len(train))
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    Xtr = feats_train(train.iloc[tr_op], ref); Xva = feats_apply(train.iloc[va_op], ref)
    yt = y_all[tr_op]
    oof_A[va_op] = make_cat().fit(Xtr, yt).predict_proba(Xva)[:, 1]
    oof_D[va_op] = make_mlp().fit(clean(Xtr), yt).predict_proba(clean(Xva))[:, 1]
    print("fold ok")

def ap_last(oof):
    return [evaluate_ap(y_all[va[op03[va]]], oof[va[op03[va]]]) for _, va in folds_full]
def show(name, oof):
    pf = ap_last(oof)
    print(f"{name:14s} recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f} | LB~ {pf[-1]-0.004:.4f}")

show("A CatBoost", oof_A)
show("D MLP", oof_D)
print("\ncorrélation A-D (op_03) :", round(np.corrcoef(oof_A[op03], oof_D[op03])[0, 1], 3))

blend = np.zeros(len(train))
for _, va in folds_full:
    vo = va[op03[va]]; blend[vo] = rank_average([oof_A[vo], oof_D[vo]])
show("blend A+D", blend)
# blend pondéré 0.7 A / 0.3 D
blend2 = np.zeros(len(train))
for _, va in folds_full:
    vo = va[op03[va]]
    ra = np.argsort(np.argsort(oof_A[vo])) / (len(vo) - 1)
    rd = np.argsort(np.argsort(oof_D[vo])) / (len(vo) - 1)
    blend2[vo] = 0.7 * ra + 0.3 * rd
show("blend 0.7A+0.3D", blend2)

## Soumission du blend (si > 0.3607 en last fold)

In [ ]:
W_A = 0.5  # blend équipondéré = meilleur en CV (last 0.3617)
ref_full = train.iloc[np.where(op03)[0]]; yf = y_all[op03]
Xf = feats_train(ref_full, ref_full)
te_op = op03_mask(test).to_numpy(); test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full)

pa = make_cat().fit(Xf, yf).predict_proba(Xte)[:, 1]
pd_ = make_mlp().fit(clean(Xf), yf).predict_proba(clean(Xte))[:, 1]
ra = np.argsort(np.argsort(pa)) / (len(pa) - 1)
rd = np.argsort(np.argsort(pd_)) / (len(pd_) - 1)
proba = W_A * ra + (1 - W_A) * rd

full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "12_cat_mlp_blend")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))